In [5]:
import re
import random
import string
from pathlib import Path

# ===== 你可以在這裡指定 conf 檔案路徑 =====
CONF_PATHS = [
    "cu_gnb.conf",            # 之前的 cu_gnb
    "du_gnb.conf",            # 之前的 du_gnb
]
OUTPUT_DIR = "./incorrect_confs"  # 修改後的檔案輸出目錄
NUM_CHANGES = 2                   # 每份 conf 隨機修改幾行

# ===== 工具函式 =====
def rand_string(n=10):
    """產生隨機字串"""
    return "".join(random.choice(string.ascii_lowercase) for _ in range(n))

def mutate_value(val: str) -> str:
    """依據參數型態產生錯誤值"""
    if re.match(r'^".*"$', val) or re.match(r"^'.*'$", val):
        return f"\"{rand_string(random.randint(6,12))}\""
    if re.match(r"^0x[0-9a-fA-F]+$", val):  # 十六進制
        return "0xDEADBEEF"
    if re.match(r"^-?\d+$", val):  # 整數
        return str(random.choice([99999, -1, -123456]))
    if re.match(r'^\(.*\)$', val):  # tuple 或 list
        return '( "nia5","nea9" )'
    return f"\"{rand_string()}\""   # 預設亂數字串

def random_modify_conf(input_path: str, output_path: str = None, num_changes: int = 1):
    """隨機修改 conf 檔案中的參數值"""
    lines = Path(input_path).read_text(encoding="utf-8").splitlines()

    # 找出候選行（含 "=" 且以 ";" 結尾）
    candidates = [i for i, line in enumerate(lines) if "=" in line and line.strip().endswith(";")]
    if not candidates:
        print(f"⚠️ {input_path} 找不到可修改的參數行")
        return

    chosen_idxs = random.sample(candidates, min(num_changes, len(candidates)))

    for idx in chosen_idxs:
        original = lines[idx]
        m = re.match(r"^\s*([A-Za-z0-9_]+)\s*=\s*(.*?);\s*$", original)
        if not m:
            continue
        key, value = m.group(1), m.group(2).strip()
        new_value = mutate_value(value)
        new_line = f"{key} = {new_value};"
        print("=== 隨機選擇的原始行 ===")
        print(original)
        print("=== 隨機修改後的行 ===")
        print(new_line, "\n")
        lines[idx] = new_line

    # 寫入檔案
    if output_path:
        Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
        Path(output_path).write_text("\n".join(lines), encoding="utf-8")
        print(f"✅ 已輸出修改後的檔案到 {output_path}")

# ===== 主程式 =====
if __name__ == "__main__":
    for conf in CONF_PATHS:
        path = Path(conf)
        if not path.exists():
            print(f"⚠️ 找不到檔案 {conf}，跳過")
            continue
        out_path = Path(OUTPUT_DIR) / f"{path.stem}_incorrect.conf"
        print(f"\n處理檔案: {conf}")
        random_modify_conf(str(path), str(out_path), NUM_CHANGES)



處理檔案: cu_gnb.conf
=== 隨機選擇的原始行 ===
  pdcp_log_level                        ="info";
=== 隨機修改後的行 ===
pdcp_log_level = "jmpjef"; 

=== 隨機選擇的原始行 ===
    remote_s_address = "127.0.0.3";
=== 隨機修改後的行 ===
remote_s_address = "yxcaaltanxo"; 

✅ 已輸出修改後的檔案到 incorrect_confs\cu_gnb_incorrect.conf

處理檔案: du_gnb.conf
=== 隨機選擇的原始行 ===
        dl_offstToCarrier                                              = 0;
=== 隨機修改後的行 ===
dl_offstToCarrier = 99999; 

=== 隨機選擇的原始行 ===
        p0_NominalWithGrant                                         =-90;
=== 隨機修改後的行 ===
p0_NominalWithGrant = 99999; 

✅ 已輸出修改後的檔案到 incorrect_confs\du_gnb_incorrect.conf


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
逐參數測試：對 conf 中每一條 `key = value;` 參數行，單獨產生一個「只改該行」的錯誤版本，
並輸出 Excel/CSV 報表（原始參數 / 修改參數 / Key / 輸出檔名）。

Per-parameter testing: for every `key = value;` line in the conf, produce a variant file
that modifies ONLY that line, and export an Excel/CSV report (original / modified / key / output file).
"""

import re
import random
import string
from pathlib import Path

# Excel/CSV 輸出套件 / For Excel/CSV export
import pandas as pd

# ===== 檔案路徑設定 / Paths =====
CONF_PATHS = [
    "cu_gnb.conf",  # 對此檔進行逐參數測試 / Run per-parameter testing on this file
]
OUTPUT_DIR = "./incorrect_confs"   # 輸出資料夾 / Output directory

# ===== 工具函式 / Utility functions =====
def rand_string(n=10):
    """產生 a-z 的隨機字串 / Generate a random lowercase string"""
    return "".join(random.choice(string.ascii_lowercase) for _ in range(n))

def mutate_value(val: str) -> str:
    """
    依據值型態產生『錯誤』的新值（故意不合理）/ Generate an intentionally 'incorrect' value
    based on the detected value type.
    """
    val = val.strip()

    # 字串 / Quoted string
    if re.match(r'^".*"$', val) or re.match(r"^'.*'$", val):
        return f"\"{rand_string(random.randint(6,12))}\""

    # IPv4 in ({ ipv4 = "x.x.x.x" })
    if re.match(r'^\(\{\s*ipv4\s*=\s*".*"\s*\}\)$', val):
        octets = [str(random.randint(200,250)) for _ in range(4)]
        return f'({{ ipv4 = "{ ".".join(octets) }" }})'

    # 十六進制 / Hex
    if re.match(r"^0x[0-9a-fA-F]+$", val):
        return "0xDEADBEEF"

    # 整數 / Integer
    if re.match(r"^-?\d+$", val):
        return str(random.choice([99999, -1, -123456, 1234567]))

    # tuple 或 list（如演算法列）/ Tuple or list (e.g., algorithms)
    if re.match(r'^\(.*\)$', val):
        return '( "nia5","nea9" )'

    # 其他 → 隨機字串 / Fallback → random string
    return f"\"{rand_string()}\""

def is_candidate_line(line: str) -> bool:
    """
    候選行：符合 `key = value;`（允許分號後面有註解；寫回時不保留註解）
    Candidate line: must match `key = value;` (allow trailing inline comment; we drop it on write).
    """
    s = line.strip()
    if not s or s.startswith("#"):
        return False
    return bool(re.match(r"^\s*[A-Za-z0-9_]+\s*=\s*.*;\s*(#.*)?$", s))

def parse_kv(line: str):
    """
    解析單行 `key = value;`，丟掉註解，只回傳 (indent, key, value, semi)
    Parse a single `key = value;` line. Ignore trailing comments.
    Return (indent, key, value, semi).
    """
    m = re.match(r"^(\s*)([A-Za-z0-9_]+)\s*=\s*(.*?)(;)", line)
    if not m:
        return None
    indent, key, value, semi = m.group(1), m.group(2), m.group(3), m.group(4)
    return indent, key, value, semi

def modify_each_param_separately(input_path: str):
    """
    對 conf 裡每個候選參數行，各自產出一份只改該行的檔案，
    並彙整 Excel / CSV 報表。

    For each candidate param line, produce a variant file with ONLY that line modified,
    and build an Excel/CSV report.
    """
    text = Path(input_path).read_text(encoding="utf-8")
    lines = text.splitlines(keepends=True)

    # 收集候選行 / Collect candidate lines
    candidates = [i for i, line in enumerate(lines) if is_candidate_line(line)]
    if not candidates:
        print(f"⚠️ {input_path} 找不到候選行 / No candidate lines found")
        return

    out_dir = Path(OUTPUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)

    successes, failures = [], []
    report_rows = []  # Excel/CSV rows
    width = len(str(len(candidates)))

    for seq, idx in enumerate(candidates, start=1):
        original_line = lines[idx]
        parsed = parse_kv(original_line)
        if not parsed:
            failures.append((idx, original_line.strip(), "parse-failed"))
            continue

        indent, key, value, semi = parsed
        new_value = mutate_value(value)
        new_line = f"{indent}{key} = {new_value}{semi}\n"  # 不保留註解 / drop inline comment

        if new_line.strip() == original_line.strip():
            failures.append((idx, original_line.strip(), "no-change"))
            continue

        # 覆蓋該行，其他內容保持不變 / Replace that line, keep others identical
        new_lines = lines.copy()
        new_lines[idx] = new_line

        # 輸出檔名 / Output filename
        safe_key = re.sub(r'[^A-Za-z0-9_]+', '_', key)[:40]
        out_path = out_dir / f"{Path(input_path).stem}_param_{str(seq).zfill(width)}_{safe_key}_incorrect.conf"
        out_path.write_text("".join(new_lines), encoding="utf-8")

        print(f"[OK] ({seq}/{len(candidates)}) {key}")
        print("  原始 / Original：", original_line.rstrip())
        print("  修改 / Modified：", new_line.rstrip())

        successes.append((idx, key, out_path.name))
        report_rows.append({
            "Key": key,
            "原始參數 / Original": original_line.strip(),
            "修改參數 / Modified": new_line.strip(),
            "輸出檔名 / Output File": out_path.name,
        })

    # 總結 / Summary
    print("\n==== 測試完成 / Done ====")
    print(f"總候選行數 / Total candidates：{len(candidates)}")
    print(f"成功修改 / Modified successfully：{len(successes)}")
    print(f"失敗 / Failed：{len(failures)}")
    if failures:
        print("\n失敗清單 / Failures：")
        for idx, line, reason in failures:
            print(f"- 行 {idx+1}: {reason} | {line}")

    # 匯出報表 / Export report
    if report_rows:
        df = pd.DataFrame(report_rows, columns=["Key", "原始參數 / Original", "修改參數 / Modified", "輸出檔名 / Output File"])
        base = Path(input_path).stem
        xlsx_path = out_dir / f"{base}_mod_report.xlsx"
        csv_path  = out_dir / f"{base}_mod_report.csv"
        df.to_excel(xlsx_path, index=False)
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")
        print(f"\n🧾 已輸出報表 / Reports saved:\n- {xlsx_path}\n- {csv_path}")

# ===== 進入點 / Entry point =====
if __name__ == "__main__":
    for conf in CONF_PATHS:
        path = Path(conf)
        if not path.exists():
            print(f"⚠️ 找不到檔案 / File not found: {conf}")
            continue
        print(f"\n[逐參數測試 / Per-parameter test] 檔案 / File：{conf}")
        modify_each_param_separately(str(path))



[逐參數測試] 檔案：cu_gnb.conf
[OK] (1/37) Asn1_verbosity
  原始： Asn1_verbosity = "none";
  修改： Asn1_verbosity = "nbjqgecurejy";
[OK] (2/37) Num_Threads_PUSCH
  原始： Num_Threads_PUSCH = 8;
  修改： Num_Threads_PUSCH = 99999;
[OK] (3/37) gNB_ID
  原始：     gNB_ID = 0xe00;
  修改：     gNB_ID = 0xDEADBEEF;
[OK] (4/37) gNB_name
  原始：     gNB_name  =  "gNB-Eurecom-CU";
  修改：     gNB_name = "vvxafdupszgr";
[OK] (5/37) tracking_area_code
  原始：     tracking_area_code  =  1;
  修改：     tracking_area_code = 1234567;
[OK] (6/37) mcc
  原始：                 mcc = 001;
  修改：                 mcc = 1234567;
[OK] (7/37) mnc
  原始：                 mnc = 01;
  修改：                 mnc = 99999;
[OK] (8/37) mnc_length
  原始：                 mnc_length = 2;
  修改：                 mnc_length = -123456;
[OK] (9/37) sst
  原始：                   sst = 1;
  修改：                   sst = 1234567;
[OK] (10/37) nr_cellid
  原始：     nr_cellid = 1;
  修改：     nr_cellid = 99999;
[OK] (11/37) tr_s_preference
  原始：     tr_s_preference = "f1";
  修